## Assemble Pseudobulk .h5ad

In this notebook, we'll conver the .csv-formatted pseudobulk data to .h5ad for release and re-use.

In [33]:
from datetime import date

import anndata
import hisepy
import os
import numpy as np
import pandas as pd
import polars as pl
import scanpy as sc
import scipy.sparse as scs
import re

In [2]:
if not os.path.isdir('output'):
    os.mkdir('output')

In [3]:
def element_id(n = 3):
    import periodictable
    from random import randrange
    rand_el = []
    for i in range(n):
        el = randrange(0,118)
        rand_el.append(periodictable.elements[el].name)
    rand_str = '-'.join(rand_el)
    return rand_str

In [4]:
csv_uuids = {
    'meta': 'a2d2c904-3a77-4f46-8faf-f06806724d52',
    'sum': '78f8c7e0-cd2a-48c2-9618-74f151ebf386',
    'det': '447bfca2-9793-4afd-84e8-726938cd29a3',
    'mean': '0be7945d-08f2-4d9c-a2e3-c0e2b8413be4'
}

In [5]:
csv_files = {}
for name, uuid in csv_uuids.items():
    csv = hisepy.cache_files([uuid])[0]
    csv_files[name] = csv

#### Metadata

In [6]:
meta = pl.read_csv(csv_files['meta'])

In [49]:
meta.head()

cohort.cohortGuid,subject.ageAtFirstDraw,subject.biologicalSex,subject.birthYear,subject.bmi,subject.cmv,subject.ethnicity,subject.race,subject.subjectGuid,sample.drawYear,sample.sampleKitGuid,sample.subjectAgeAtDraw,sample.visitName,specimen.specimenGuid,batch_id,pool_id,AIFI_L1,AIFI_L2,AIFI_L3,n_cells,barcodes
str,i64,str,i64,i64,str,str,str,str,i64,str,i64,str,str,str,str,str,str,str,i64,str
"""BR1""",32,"""Female""",1987,23,"""Negative""","""Non-Hispanic origin""","""Caucasian""","""BR1001""",2019,"""KT00001""",32,"""Flu Year 1 Day 0""","""PB00001-01""","""B001""","""B001-P1""","""DC""","""ASDC""","""ASDC""",7,"""BR1001_Flu-Year-1-Day-0_ASDC"""
"""BR1""",32,"""Female""",1987,23,"""Negative""","""Non-Hispanic origin""","""Caucasian""","""BR1001""",2019,"""KT00001""",32,"""Flu Year 1 Day 0""","""PB00001-01""","""B001""","""B001-P1""","""B cell""","""Memory B cell""","""Activated memory B cell""",3,"""BR1001_Flu-Year-1-Day-0_Activa…"
"""BR1""",32,"""Female""",1987,23,"""Negative""","""Non-Hispanic origin""","""Caucasian""","""BR1001""",2019,"""KT00001""",32,"""Flu Year 1 Day 0""","""PB00001-01""","""B001""","""B001-P1""","""NK cell""","""CD56dim NK cell""","""Adaptive NK cell""",99,"""BR1001_Flu-Year-1-Day-0_Adapti…"
"""BR1""",32,"""Female""",1987,23,"""Negative""","""Non-Hispanic origin""","""Caucasian""","""BR1001""",2019,"""KT00001""",32,"""Flu Year 1 Day 0""","""PB00001-01""","""B001""","""B001-P1""","""Progenitor cell""","""Progenitor cell""","""BaEoMaP cell""",1,"""BR1001_Flu-Year-1-Day-0_BaEoMa…"
"""BR1""",32,"""Female""",1987,23,"""Negative""","""Non-Hispanic origin""","""Caucasian""","""BR1001""",2019,"""KT00001""",32,"""Flu Year 1 Day 0""","""PB00001-01""","""B001""","""B001-P1""","""Monocyte""","""CD16 monocyte""","""C1Q+ CD16 monocyte""",117,"""BR1001_Flu-Year-1-Day-0_C1Qpos…"


#### Count sums

In [7]:
sums = pl.read_csv(csv_files['sum'], infer_schema_length = 100000)

In [8]:
sums_row_names = sums['gene'].to_list()
sums_col_names = sums.columns

In [9]:
sums = sums.drop('gene').to_numpy()

In [10]:
sums = scs.csr_matrix(sums)

In [11]:
sums = sums.astype(np.uint32)

In [12]:
sums

<Compressed Sparse Row sparse matrix of dtype 'uint32'
	with 553606552 stored elements and shape (33538, 58816)>

#### Means of Normalized values

In [13]:
means = pl.read_csv(csv_files['mean'], infer_schema_length = 100000)

In [14]:
means_row_names = means['gene'].to_list()
means_col_names = means.columns

In [15]:
means = means.drop('gene').to_numpy()

In [16]:
means = scs.csr_matrix(means)

In [17]:
means = means.astype(np.float32)

In [18]:
means

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 553606552 stored elements and shape (33538, 58816)>

#### Detection counts

In [19]:
det = pl.read_csv(csv_files['det'], infer_schema_length = 100000)

In [20]:
det_row_names = det['gene'].to_list()
det_col_names = det.columns

In [21]:
det = det.drop('gene').to_numpy()

In [22]:
det = scs.csr_matrix(det)

In [27]:
det = det.astype(np.uint16)

In [28]:
det

<Compressed Sparse Row sparse matrix of dtype 'uint16'
	with 553606552 stored elements and shape (33538, 58816)>

## Build AnnData object

In [31]:
obs = meta.to_pandas()
obs = obs.set_index('barcodes', drop = True)

In [35]:
adata = anndata.AnnData(
    obs = obs,
    layers = {
        'sum': sums.transpose(),
        'means': means.transpose(),
        'detect': det.transpose()
    }
)

In [37]:
adata.var_names = sums_row_names

In [40]:
out_h5ad = 'output/sound-life_AIFI_L3_pseudobulk_{d}.h5ad'.format(d = date.today())
adata.write_h5ad(out_h5ad)

## Upload .h5ad data to HISE

Finally, we'll use `hisepy.upload.upload_files()` to send a copy of our output to HISE to use for downstream analysis steps.

In [41]:
study_space_uuid = 'de025812-5e73-4b3c-9c3b-6d0eac412f2a'
title = 'Sound Life Pseudobulk .h5ad {d}'.format(d = date.today())

In [43]:
search_id = element_id()
search_id

'nobelium-erbium-copernicium'

In [44]:
in_files = list(csv_uuids.values())
in_files

['a2d2c904-3a77-4f46-8faf-f06806724d52',
 '78f8c7e0-cd2a-48c2-9618-74f151ebf386',
 '447bfca2-9793-4afd-84e8-726938cd29a3',
 '0be7945d-08f2-4d9c-a2e3-c0e2b8413be4']

In [45]:
out_files = [out_h5ad]
out_files

['output/sound-life_AIFI_L3_pseudobulk_2025-12-09.h5ad']

In [46]:
len(out_files)

1

In [47]:
import session_info
session_info.show()

/home/workspace/environment/pythonscrna12/lib/python3.13/site-packages/session_info/main.py:213: UserWarning: The '__version__' attribute is deprecated and will be removed in MarkupSafe 3.1. Use feature detection, or `importlib.metadata.version("markupsafe")`, instead.
  mod_version = _find_version(mod.__version__)


In [50]:
hisepy.upload.upload_files(
    files = out_files,
    study_space_id = study_space_uuid,
    title = title,
    input_file_ids = in_files,
    destination = search_id
)

Please provide input of comma separated sample ids for the files being uploaded:  


{'Message': 'General Okay-ness',
 'VisualizationId': '00000000-0000-0000-0000-000000000000',
 'AbstractionId': '00000000-0000-0000-0000-000000000000',
 'TraceId': '1bbbff90-65f7-4034-9d21-0efd391c87d7',
 'ProcessId': 'a336afbe-29ed-42b8-9a03-23e89ffc00b4',
 'WorkflowId': '01fa0151-461f-45ef-9029-bbf3d413dfa3',
 'FileIds': ['68c779c4-62d9-447b-8db1-a046a06d2bb1']}